<a href="https://colab.research.google.com/github/x1001000/Colab-Notebooks/blob/main/MP3_to_SRT_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 輸入 OpenAI API 金鑰：Ken的，勿外流

In [ ]:
import getpass
import os
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")
print("API key set successfully!")

# 上傳MP3：左側面板 > 檔案 > 上傳按鈕

In [ ]:
import glob
mp3_files = glob.glob('*.mp3')
if mp3_files:
    if len(mp3_files) > 1:
        for i, mp3_file in enumerate(mp3_files):
            print(f'#{i+1} {mp3_file}')
        mp3_filename = mp3_files[int(input('選一個MP3 #'))-1]
    else:
        mp3_filename = mp3_files[0]
    print(f'MP3 👉 {mp3_filename}')
else:
    print("請先上傳MP3，然後再執行一次")

# 安裝套件：音檔切塊、OpenAI串接、簡繁轉換

In [ ]:
!pip install pydub openai OpenCC --quiet

# 生成SRT：使用 OpenAI Whisper

In [ ]:
import openai
from pydub import AudioSegment
import os
import re
from opencc import OpenCC
cc = OpenCC('s2tw')

def chunk_audio(filename, chunk_length_ms=20*60*1000):  # 20 minutes per chunk
    """Split audio into chunks if larger than 25MB"""
    audio = AudioSegment.from_mp3(filename)
    file_size_mb = os.path.getsize(filename) / (1024 * 1024)

    if file_size_mb <= 25:
        return [filename], [0]

    chunks = []
    start_times = []

    for i, start in enumerate(range(0, len(audio), chunk_length_ms)):
        chunk = audio[start:start + chunk_length_ms]
        chunk_filename = f"chunk_{i}.mp3"
        chunk.export(chunk_filename, format="mp3")
        chunks.append(chunk_filename)
        start_times.append(start / 1000)  # Convert to seconds

    return chunks, start_times

def transcribe_audio(filename):
    """Transcribe audio using OpenAI Whisper"""
    client = openai.OpenAI()

    with open(filename, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="srt"
        )

    return transcript

def adjust_srt_timestamps(srt_content, offset_seconds):
    """Adjust SRT timestamps by adding offset"""
    if offset_seconds == 0:
        return srt_content

    def adjust_timestamp(match):
        timestamp = match.group(0)
        # Parse timestamp: HH:MM:SS,mmm
        time_parts = timestamp.replace(',', ':').split(':')
        hours, minutes, seconds, milliseconds = map(int, time_parts)

        total_seconds = hours * 3600 + minutes * 60 + seconds + milliseconds / 1000.0
        total_seconds += offset_seconds

        new_hours = int(total_seconds // 3600)
        new_minutes = int((total_seconds % 3600) // 60)
        new_seconds = int(total_seconds % 60)
        new_milliseconds = int((total_seconds % 1) * 1000)

        return f"{new_hours:02d}:{new_minutes:02d}:{new_seconds:02d},{new_milliseconds:03d}"

    # Regex to match timestamp format: HH:MM:SS,mmm
    timestamp_pattern = r'\d{2}:\d{2}:\d{2},\d{3}'
    return re.sub(timestamp_pattern, adjust_timestamp, srt_content)

def merge_srt_files(srt_contents, start_times):
    """Merge multiple SRT contents with proper indexing and timestamps"""
    merged_content = []
    subtitle_index = 1

    for srt_content, start_offset in zip(srt_contents, start_times):
        # Adjust timestamps for this chunk
        adjusted_content = adjust_srt_timestamps(srt_content, start_offset)

        # Split into blocks and reindex
        blocks = adjusted_content.strip().split('\n\n')

        for block in blocks:
            if block.strip():
                lines = block.strip().split('\n')
                if len(lines) >= 3:  # Valid SRT block
                    # Replace the index with our sequential one
                    lines[0] = str(subtitle_index)
                    merged_content.append('\n'.join(lines))
                    subtitle_index += 1

    return '\n\n'.join(merged_content)

# Main processing
print("Processing audio file...")

# Chunk the audio if necessary
chunks, start_times = chunk_audio(mp3_filename)
print(f"Audio split into {len(chunks)} chunk(s)")

# Transcribe each chunk
srt_contents = []
for i, chunk in enumerate(chunks):
    print(f"Transcribing chunk {i+1}/{len(chunks)}...")
    srt_content = transcribe_audio(chunk)
    srt_contents.append(srt_content)

    # Clean up chunk files if we created them
    if chunk != mp3_filename:
        os.remove(chunk)

# Merge SRT contents
srt_filename = os.path.splitext(mp3_filename)[0] + ".srt"
if len(srt_contents) == 1:
    # Single file, no need to merge
    final_srt = srt_contents[0]
else:
    # Multiple chunks, merge with timestamp adjustment
    final_srt = merge_srt_files(srt_contents, start_times)

# Write final SRT file
with open(srt_filename, 'w', encoding='utf-8') as f:
    f.write(cc.convert(final_srt))

print(f"SRT file created: {srt_filename}")
print("Transcription complete!")

# 下載SRT

In [ ]:
print(f"SRT 👉 {srt_filename}")
from google.colab import files
files.download(srt_filename)
print("Download complete!")